# Обучение моделей cat/dog — оркестратор для ВМ

Запускать **из корня проекта** на ВМ. Каждая ячейка зовёт CLI-шаги из README.

## Подготовка (один раз, в терминале ВМ)

```bash
python3.10 -m venv ~/practicum_venv
source ~/practicum_venv/bin/activate
cd ~/nn_cv_sprint2_full
bash practicum_work/setup_vm.sh
clearml-init
```

Затем откройте этот ноутбук в **VSCode Remote-SSH** и выберите **kernel = `~/practicum_venv`**.

## Что внутри ноутбука
1. sanity-check конфига бейзлайна;
2. обучение бейзлайна — DeepLabV3+ R50-d8;
3. анализ качества лучшего чекпойнта на test.

UNet «с нуля» по обоснованию из учебника (см. README Этап 2) сознательно
не запускается. Эксперименты Этапа 3 (аугментации, лосс 1:3, R101-mg124)
добавляются ниже после получения метрики бейзлайна.

In [ ]:
# cwd ядра ставим в корень проекта (VSCode по умолчанию открывает с cwd = папка ноутбука)
%cd ~/nn_cv_sprint2_full

## 1. Sanity-check конфига бейзлайна (без обучения)

In [ ]:
!python practicum_work/sanity_check.py configs/deeplabv3plus_practice/deeplabv3plus_r50-d8_1xb16-practice_dataset-256x256.py

## 2. Обучение бейзлайна — DeepLabV3+ R50-d8

Ожидаемое время — **~30–45 мин на одном T4**: из таблицы замеров скорости в
уроке «Современные модификации» для `deeplabv3plus_r50-d8` это ~0.077 c/итер,
у нас ~1200 итер (100 эпох × ~12 итер/эпоху при batch=16).

Целевая метрика проекта — **mDice > 0.75** на test. ClearML-ссылка
появится в первых строках вывода ниже.

In [ ]:
!python tools/train.py configs/deeplabv3plus_practice/deeplabv3plus_r50-d8_1xb16-practice_dataset-256x256.py

## 3. Per-image Dice анализ лучшего чекпойнта бейзлайна на test

Скрипт сохраняет CSV `per_image_dice.csv` плюс по 5 best/worst PNG-триплетов
(image | GT | pred) для отчёта Этапа 2.

In [ ]:
import glob

CFG  = "configs/deeplabv3plus_practice/deeplabv3plus_r50-d8_1xb16-practice_dataset-256x256.py"
NAME = CFG.split("/")[-1].replace(".py", "")
ckpts = sorted(glob.glob(f"work_dirs/{NAME}/best_mDice_epoch_*.pth"))
assert ckpts, f"no best checkpoint for {NAME}"
ckpt = ckpts[-1]
print("==>", NAME, ckpt)

!python practicum_work/src/analysis/per_image_dice.py \
    --config {CFG} \
    --checkpoint {ckpt} \
    --split test \
    --out practicum_work/supplementary/viz/test_baseline_deeplab --n 5